In [1]:
pip install yfinance

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm


def obtener_variacion_logaritmica_porcentaje(ticker):
    """
    Esta función devuelve un DataFrame con las variaciones logarítmicas de los precios de cierre
    de un ticker dado para un rango de fechas, expresadas en porcentaje y con el formato decimal ajustado
    para usar comas como separadores decimales, además de añadir el símbolo de porcentaje.

    Parámetros:
    - ticker: El símbolo del ticker de la acción (como string).
    - fecha_inicio: La fecha de inicio del rango en formato 'AAAA-MM-DD'.
    - fecha_fin: La fecha de fin del rango en formato 'AAAA-MM-DD'.
    """
    fecha_inicio = "2020-01-01"
    fecha_fin = "2023-12-31"
    # Descargar los datos del ticker
    datos = yf.download(ticker, start=fecha_inicio, end=fecha_fin)

    # Seleccionar solo la columna 'Close'
    precios_cierre = datos[['Close']]

    # Calcular la variación logarítmica de los precios de cierre
    variacion_log = np.log(precios_cierre / precios_cierre.shift(1))

    # Convertir la variación logarítmica a formato porcentual
    variacion_log_porcentaje = variacion_log * 100

    # Convertir a string, usar coma como separador decimal y añadir el símbolo de porcentaje
    variacion_log_porcentaje = variacion_log_porcentaje['Close'].replace('.', ',')

    # Crear un nuevo DataFrame para devolver, usando la fecha como índice
    df_resultado = pd.DataFrame(variacion_log_porcentaje)
    df_resultado.rename(columns={'Close': 'Variacion Logaritmica (%)'}, inplace=True)

    return df_resultado



In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm

def regresion(ticker):
    # Suponiendo que tienes una función obtener_variacion_logaritmica_porcentaje que recoge los datos y realiza el cálculo
    df_variacion_log = obtener_variacion_logaritmica_porcentaje(ticker)

    # Cargando un DataFrame externo que supongamos contiene datos como 'Mkt-RF', 'SMB', 'HML', etc.
    famafrench_df = pd.read_csv('/content/csv_general.csv', sep=';')
    famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'])
    famafrench_df.set_index('Date', inplace=True)

    # Combinar los DataFrames
    df_combinado = famafrench_df.merge(df_variacion_log, left_index=True, right_index=True, how='outer')
    df_combinado.dropna(inplace=True)
    df_combinado.round(3)

    # Ajustando y formateando los datos como se necesita
    df_combinado['Mkt-RF'] = df_combinado['Mkt-RF'].astype(str).str.replace(',', '.').astype(float)
    df_combinado['SMB'] = df_combinado['SMB'].astype(str).str.replace(',', '.').astype(float)
    df_combinado['HML'] = df_combinado['HML'].astype(str).str.replace(',', '.').astype(float)
    df_combinado['RF'] = df_combinado['RF'].astype(str).str.rstrip('%').str.replace(',', '.').astype(float) / 100
    df_combinado['Variacion Logaritmica (%)'] = df_combinado['Variacion Logaritmica (%)'].astype(str).str.rstrip('%').str.replace(',', '.').astype(float) / 100
    df_combinado['% Fundflows'] = df_combinado['% Fundflows'].astype(str).str.rstrip('%').str.replace(',', '.').astype(float) / 100

    # Preparando variables para la regresión
    X = df_combinado[['Mkt-RF', 'SMB', 'HML', '% Fundflows']]
    X = sm.add_constant(X)
    Y = df_combinado['Variacion Logaritmica (%)'] - df_combinado['RF']

    # Ajustar modelo OLS
    modelo = sm.OLS(Y, X).fit()

    # Retornar el p-valor de '% Fundflows'
    p_valor_fundflows = modelo.pvalues['% Fundflows']

    return p_valor_fundflows



In [4]:
# Asume que esta es tu lista de tickers
lista_tickers = ["MSFT",
"AAPL",
"NVDA",
"AMZN",
"META",
"GOOGL",
"GOOG",
"BRK.B",
"LLY",
"AVGO",
"JPM",
"TSLA",
"XOM",
"V",
"UNH",
"MA",
"PG",
"JNJ",
"HD",
"MRK",
"COST",
"ABBV",
"CRM",
"CVX",
"AMD",
"NFLX",
"BAC",
"WMT",
"PEP",
"KO",
"LIN",
"TMO",
"ADBE",
"DIS",
"ACN",
"WFC",
"ORCL",
"CSCO",
"MCD",
"QCOM",
"ABT",
"CAT",
"INTU",
"AMAT",
"IBM",
"VZ",
"GE",
"CMCSA",
"NOW",
"INTC",
"DHR",
"COP",
"UBER",
"TXN",
"PFE",
"UNP",
"AMGN",
"PM",
"LOW",
"SPGI",
"ISRG",
"MU",
"RTX",
"GS",
"NEE",
"HON",
"ETN",
"AXP",
"LRCX",
"BKNG",
"PGR",
"T",
"ELV",
"SYK",
"C",
"MS",
"PLD",
"BLK",
"MDT",
"TJX",
"NKE",
"UPS",
"SCHW",
"DE",
"CI",
"BA",
"VRTX",
"BMY",
"CB",
"ADP",
"MMC",
"BSX",
"REGN",
"SBUX",
"ADI",
"LMT",
"FI",
"KLAC",
"CVS",
"BX",
"MDLZ",
"AMT",
"SNPS",
"GILD",
"PANW",
"CDNS",
"TMUS",
"CMG",
"MPC",
"EOG",
"ICE",
"TGT",
"SHW",
"SLB",
"CME",
"SO",
"ZTS",
"WM",
"ANET",
"DUK",
"MO",
"EQIX",
"PH",
"PSX",
"CL",
"ITW",
"FCX",
"PYPL",
"CSX",
"BDX",
"MCK",
"ABNB",
"APH",
"TT",
"TDG",
"USB",
"GD",
"ORLY",
"EMR",
"HCA",
"NOC",
"PNC",
"PCAR",
"AON",
"FDX",
"PXD",
"NXPI",
"MAR",
"MCO",
"VLO",
"CEG",
"CTAS",
"MSI",
"ROP",
"ECL",
"NSC",
"EW",
"COF",
"AIG",
"DXCM",
"HLT",
"AZO",
"APD",
"F",
"TRV",
"AJG",
"ADSK",
"TFC",
"GM",
"WELL",
"MMM",
"NUE",
"SPG",
"CPRT",
"CARR",
"MCHP",
"URI",
"ROST",
"WMB",
"DHI",
"SMCI",
"OKE",
"PSA",
"NEM",
"OXY",
"MET",
"AFL",
"ALL",
"TEL",
"GWW",
"SRE",
"O",
"AEP",
"IQV",
"JCI",
"AMP",
"FTNT",
"CCI",
"MSCI",
"DLR",
"FAST",
"FIS",
"BK",
"HES",
"STZ",
"IDXX",
"KMB",
"A",
"DOW",
"AME",
"PRU",
"LULU",
"LEN",
"MNST",
"CMI",
"D",
"CTVA",
"ODFL",
"OTIS",
"COR",
"PAYX",
"LHX",
"GIS",
"HUM",
"CNC",
"SYY",
"RSG",
"MLM",
"CSGP",
"PWR",
"IR",
"YUM",
"EXC",
"GEHC",
"FANG",
"IT",
"HAL",
"KR",
"PCG",
"VMC",
"CTSH",
"KMI",
"GEV",
"ACGL",
"MRNA",
"KVUE",
"DG",
"BKR",
"DVN",
"CDW",
"EL",
"ADM",
"GPN",
"PEG",
"PPG",
"VRSK",
"DD",
"RCL",
"MPWR",
"ROK",
"KDP",
"EA",
"EFX",
"EXR",
"DFS",
"ED",
"HIG",
"VICI",
"FICO",
"XYL",
"DAL",
"ANSS",
"XEL",
"BIIB",
"FTV",
"ON",
"KHC",
"HSY",
"WST",
"CBRE",
"MTD",
"KEYS",
"WTW",
"RMD",
"EIX",
"CHTR",
"TSCO",
"CAH",
"WAB",
"EBAY",
"DLTR",
"ZBH",
"LYB",
"TROW",
"AVB",
"HWM",
"TRGP",
"WEC",
"HPQ",
"WY",
"NVR",
"CHD",
"PHM",
"BLDR",
"FITB",
"DOV",
"GLW",
"RJF",
"TTWO",
"BR",
"NDAQ",
"STT",
"WDC",
"MTB",
"HPE",
"AWK",
"IRM",
"SBAC",
"GRMN",
"ALGN",
"DECK",
"DTE",
"STLD",
"ETR",
"HUBB",
"ULTA",
"PTC",
"MOH",
"CPAY",
"NTAP",
"AXON",
"EQR",
"IFF",
"APTV",
"BAX",
"GPC",
"CTRA",
"STE",
"BALL",
"ES",
"ILMN",
"INVH",
"BRO",
"PPL",
"HBAN",
"WAT",
"FE",
"ARE",
"COO",
"TDY",
"LVS",
"CBOE",
"VLTO",
"FSLR",
"CINF",
"AEE",
"TXT",
"MKC",
"RF",
"WBD",
"DRI",
"PFG",
"J",
"OMC",
"NTRS",
"HOLX",
"IEX",
"CLX",
"CNP",
"LH",
"JBL",
"WRB",
"LDOS",
"AVY",
"EXPE",
"SYF",
"DPZ",
"TYL",
"VTR",
"MAS",
"ATO",
"CMS",
"MRO",
"STX",
"EXPD",
"PKG",
"LUV",
"TSN",
"FDS",
"NRG",
"SWKS",
"VRSN",
"TER",
"EG",
"CE",
"CFG",
"AKAM",
"JBHT",
"CCL",
"ENPH",
"ESS",
"BBY",
"SNA",
"TRMB",
"ALB",
"BG",
"EPAM",
"MAA",
"POOL",
"CF",
"ZBRA",
"K",
"EQT",
"CAG",
"SWK",
"NDSN",
"LYV",
"DGX",
"HST",
"KEY",
"UAL",
"VTRS",
"L",
"LKQ",
"WBA",
"PNR",
"DOC",
"IP",
"AMCR",
"KMX",
"RVTY",
"CRL",
"MGM",
"ROL",
"GEN",
"JKHY",
"WRK",
"LNT",
"KIM",
"TAP",
"AES",
"EVRG",
"IPG",
"EMN",
"SJM",
"PODD",
"JNPR",
"ALLE",
"FFIV",
"HII",
"UDR",
"LW",
"QRVO",
"NI",
"CPT",
"TECH",
"APA",
"AOS",
"BBWI",
"MOS",
"UHS",
"CTLT",
"INCY",
"TFX",
"WYNN",
"HRL",
"TPR",
"PAYC",
"NWSA",
"REG",
"DAY",
"AIZ",
"HSIC",
"SOLV",
"MTCH",
"GL",
"BF.B",
"CZR",
"AAL",
"BXP",
"CPB",
"MKTX",
"CHRW",
"PNW",
"GNRC",
"BWA",
"NCLH",
"RHI",
"ETSY",
"FOXA",
"BEN",
"IVZ",
"FMC",
"FRT",
"HAS",
"DVA",
"CMA",
"BIO",
"RL",
"MHK",]

# Lista para almacenar los p-valores
p_valores_fundflows = []

for ticker in lista_tickers:
    print(f"Procesando {ticker}...")
    try:
        p_valor = regresion(ticker)
        p_valores_fundflows.append(p_valor)
    except Exception as e:
        print(f"Error al procesar {ticker}: {e}")
        p_valores_fundflows.append(None)  # Asumiendo que quieres mantener la alineación con los tickers

# Ahora p_valores_fundflows contiene los p-valores de '% Fundflows' para cada ticker
print(p_valores_fundflows)

Procesando MSFT...


[*********************100%%**********************]  1 of 1 completed
<ipython-input-3-2c39d4a527dd>:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'])


Procesando AAPL...


[*********************100%%**********************]  1 of 1 completed
<ipython-input-3-2c39d4a527dd>:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'])
[*********************100%%**********************]  1 of 1 completed

Procesando NVDA...


...

Procesando CMA...


[*********************100%%**********************]  1 of 1 completed
<ipython-input-3-2c39d4a527dd>:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'])
[*********************100%%**********************]  1 of 1 completed


Procesando BIO...


<ipython-input-3-2c39d4a527dd>:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'])
[*********************100%%**********************]  1 of 1 completed
<ipython-input-3-2c39d4a527dd>:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'])
[*********************100%%**********************]  1 of 1 completed
<ipython-input-3-2c39d4a527dd>:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'])


Procesando RL...
Procesando MHK...
[0.12651658187995588, 0.251064436631803, 0.11678516328917928, 0.3153752769454235, 0.6471652238725853, 0.8141392970040576, 0.8567693732461287, None, 0.2990937542036243, 0.12636134893327589, 0.9310271905805014, 0.04366602120522295, 0.9321103766102878, 0.16853988196546285, 0.04646553669225342, 0.1865863858995574, 0.7282325886973183, 0.9211705428742618, 0.07292679099452608, 0.8164459135579683, 0.05237084033696555, 0.3911883404424491, 0.5296625346993135, 0.9665828495515512, 0.05581798272280405, 0.2593401688522479, 0.5557290760307125, 0.27397084420860335, 0.10468299530504997, 0.8999484793832752, 0.3703210628545801, 0.5798239384843786, 0.021134057735567216, 0.9015939182326486, 0.7940790024781799, 0.6729851981146537, 0.9203230441263186, 0.6770396460359734, 0.3912592404528542, 0.2456056984774852, 0.8358996134359836, 0.33165428537178565, 0.9403868335866098, 0.0763913311849959, 0.4640155296201257, 0.619266621322418, 0.6947089855073958, 0.9392571151478153, 0.4453

In [ ]:
df_combinado.to_csv('ex.csv', sep = ";", index=True)

In [5]:
print(p_valores_fundflows)

[0.12651658187995588, 0.251064436631803, 0.11678516328917928, 0.3153752769454235, 0.6471652238725853, 0.8141392970040576, 0.8567693732461287, None, 0.2990937542036243, 0.12636134893327589, 0.9310271905805014, 0.04366602120522295, 0.9321103766102878, 0.16853988196546285, 0.04646553669225342, 0.1865863858995574, 0.7282325886973183, 0.9211705428742618, 0.07292679099452608, 0.8164459135579683, 0.05237084033696555, 0.3911883404424491, 0.5296625346993135, 0.9665828495515512, 0.05581798272280405, 0.2593401688522479, 0.5557290760307125, 0.27397084420860335, 0.10468299530504997, 0.8999484793832752, 0.3703210628545801, 0.5798239384843786, 0.021134057735567216, 0.9015939182326486, 0.7940790024781799, 0.6729851981146537, 0.9203230441263186, 0.6770396460359734, 0.3912592404528542, 0.2456056984774852, 0.8358996134359836, 0.33165428537178565, 0.9403868335866098, 0.0763913311849959, 0.4640155296201257, 0.619266621322418, 0.6947089855073958, 0.9392571151478153, 0.44533665284296187, 0.08054386741853059,